<a href="https://colab.research.google.com/github/COfek/Astronomical-Image-Denoising/blob/SMD%26ADMM%26RED/astro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os
import sys
import math
import random
import json
import torch
import torch.nn as nn
import torch.fft
import torch.nn.functional as F
from torch import Tensor
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
import torchvision.utils as vutils
from PIL import Image
from pathlib import Path
from typing import Optional, Callable, Tuple, List, Dict
import zipfile
from concurrent.futures import ThreadPoolExecutor
import cv2
import numpy as np
import requests
from scipy.signal import convolve2d
from tqdm import tqdm
from threading import Lock
from skimage.restoration import denoise_tv_chambolle
from skimage.metrics import structural_similarity as ssim
import matplotlib.pyplot as plt

# Dataset Class

In [2]:
class AstroDenoisingDataset(Dataset):
    """
    Custom PyTorch Dataset for astronomical image denoising.

    This dataset loads paired grayscale images: clean targets and noisy inputs.
    The files must be PNG images stored in two parallel folders with identical filenames.

    Args:
        clean_dir (str or Path): Directory containing clean (target) images.
        noisy_dir (str or Path): Directory containing noisy (input) images.
        transform (callable, optional): Optional transformation to apply to both images (default: ToTensor).

    Returns:
        A tuple (noisy_tensor, clean_tensor) where:
            - noisy_tensor: input image with noise (shape: [1, H, W])
            - clean_tensor: ground truth image (shape: [1, H, W])
    """
    def __init__(
        self,
        clean_dir: str,
        noisy_dir: str,
        transform: Optional[Callable] = None
    ) -> None:
        self.clean_paths = sorted(Path(clean_dir).glob("*.png"))
        self.noisy_paths = sorted(Path(noisy_dir).glob("*.png"))
        assert len(self.clean_paths) == len(self.noisy_paths), "Mismatch between clean and noisy images"
        self.transform = transform or transforms.ToTensor()

    def __len__(self) -> int:
        return len(self.clean_paths)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        clean_img = Image.open(self.clean_paths[idx]).convert("L")
        noisy_img = Image.open(self.noisy_paths[idx]).convert("L")

        clean_tensor = self.transform(clean_img)
        noisy_tensor = self.transform(noisy_img)

        return noisy_tensor, clean_tensor
#==============================================================================================


# Data Utils

In [3]:
# === Configuration ===
NUM_IMAGES: int = 1000 # Number of images to simulate
OUTPUT_DIR: str = "Data" # Output directory for clean and noisy images
IMAGE_SIZE: Tuple[int, int] = (256, 256)  # Size to resize images to
GAUSSIAN_SIGMA: float = 0.05 # Standard deviation for Gaussian noise
PSF_SIGMA: float = 2 # Standard deviation for Gaussian PSF
PSF_SIZE: int = 11 # Size of the Gaussian PSF (must be odd)
VERBOSE: bool = False  # Flag to control print statements

# === Output Directories ===
clean_dir: Path = Path(OUTPUT_DIR) / "clean"
noisy_dir: Path = Path(OUTPUT_DIR) / "noisy"
clean_dir.mkdir(parents=True, exist_ok=True)
noisy_dir.mkdir(parents=True, exist_ok=True)

def gaussian_kernel(size: int = 11, sigma: float = 2) -> np.ndarray:
    """
    Generate a normalized 2D Gaussian kernel.

    Args:
        size (int): Width and height of the kernel (must be odd).
        sigma (float): Standard deviation of the Gaussian.

    Returns:
        np.ndarray: 2D Gaussian kernel normalized to sum to 1.
    """
    ax = np.linspace(-(size // 2), size // 2, size)
    xx, yy = np.meshgrid(ax, ax)
    kernel = np.exp(-(xx**2 + yy**2) / (2. * sigma**2))
    return kernel / np.sum(kernel)

psf: np.ndarray = gaussian_kernel(PSF_SIZE, PSF_SIGMA)
np.save(Path(OUTPUT_DIR) / "psf.npy", psf)

def simulate_blurred_noisy(clean_img: np.ndarray, psf: np.ndarray, sigma: float) -> np.ndarray:
    """
    Apply Gaussian blur and additive noise to a clean image.

    Args:
        clean_img (np.ndarray): Clean input image, normalized to [0, 1].
        psf (np.ndarray): Point spread function for blurring.
        sigma (float): Standard deviation of Gaussian noise.

    Returns:
        np.ndarray: Simulated noisy and blurred image.
    """
    blurred = convolve2d(clean_img, psf, mode='same', boundary='wrap')
    noise = np.random.normal(0, sigma, clean_img.shape)
    return np.clip(blurred + noise, 0, 1)

def process_image(img_bytes: bytes, idx: int) -> bool:
    """
    Process an image: decode, resize, normalize, simulate noise+blur, and save.

    Args:
        img_bytes (bytes): Raw image content as byte stream.
        idx (int): Index for filename.

    Returns:
        bool: True if processing and saving succeeded, False otherwise.
    """
    try:
        img_array = np.frombuffer(img_bytes, np.uint8)
        img = cv2.imdecode(img_array, cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise ValueError("Image decode failed")

        img = cv2.resize(img, IMAGE_SIZE)
        clean_np = img / 255.0
        noisy_np = simulate_blurred_noisy(clean_np, psf, GAUSSIAN_SIGMA)

        cv2.imwrite(str(clean_dir / f"{idx:05d}.png"), (clean_np * 255).astype(np.uint8))
        cv2.imwrite(str(noisy_dir / f"{idx:05d}.png"), (noisy_np * 255).astype(np.uint8))

        if VERBOSE:
            print(f"✅ Image {idx:05d} saved")
        return True
    except Exception as e:
        if VERBOSE:
            print(f"❌ Failed to process image {idx}: {e}")
        return False

def download_sdss_urls(n: int) -> List[str]:
    """
    Generate SDSS image URLs.

    Args:
        n (int): Number of URLs to generate.

    Returns:
        List[str]: List of SDSS image URLs.
    """
    return [
        f"https://skyserver.sdss.org/dr16/SkyServerWS/ImgCutout/getjpeg?ra={180+i}&dec=0&scale=0.2&width=256&height=256"
        for i in range(n)
    ]

def download_process_worker(url: str, idx: int, pbar: tqdm) -> bool:
    """
    Download and process a single image from a URL.

    Args:
        url (str): URL to download the image from.
        idx (int): Image index for saving.
        pbar (tqdm): Progress bar to update after each attempt.

    Returns:
        bool: True if successful, False otherwise.
    """
    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 200:
            result = process_image(response.content, idx)
            pbar.update(1)
            return result
        else:
            if VERBOSE:
                print(f"❌ HTTP {response.status_code} at {url}")
    except Exception as e:
        if VERBOSE:
            print(f"❌ Error downloading {url}: {e}")
    pbar.update(1)
    return False

def download_and_process_sdss() -> None:
    """
    Download images from SDSS and process them using multi-threading.
    Saves clean and noisy pairs to output directories.
    """
    urls = download_sdss_urls(NUM_IMAGES)
    with ThreadPoolExecutor(max_workers=16) as executor:
        futures = []
        with tqdm(total=len(urls), desc="Simulating") as pbar:
            for idx, url in enumerate(urls):
                futures.append(executor.submit(download_process_worker, url, idx, pbar))
            for f in futures:
                f.result()
    if VERBOSE:
        print(f"\n🎉 Done! {len(futures)} images saved to `{OUTPUT_DIR}`")

#  === Downsample DIV2K Dataset ===

def downsample_image(img: np.ndarray, factor: int) -> np.ndarray:
    """Downsamples an image by a given factor using area interpolation."""
    h, w = img.shape[:2]
    return cv2.resize(img, (w // factor, h // factor), interpolation=cv2.INTER_AREA)

def process_and_save(img_path: Path, downsampled_dir: Path, factor: int, pbar: tqdm, lock: Lock) -> None:
    """Reads, downsamples, and saves the high-res and low-res versions of an image."""
    img = cv2.imread(str(img_path))
    if img is not None:
        down = downsample_image(img, factor)
        cv2.imwrite(str(downsampled_dir / img_path.name), down)
    with lock:
        pbar.update(1)

#=============================================================================================


# Utils

In [4]:
#========== Metrics Computations ==========
def compute_psnr(img1: Tensor, img2: Tensor) -> float:
    """
    Compute PSNR between two images (values in [0, 1]).

    Args:
        img1 (Tensor): Shape [1, 1, H, W] or [1, H, W]
        img2 (Tensor): Shape [1, 1, H, W] or [1, H, W]

    Returns:
        float: PSNR in dB
    """
    mse = torch.mean((img1 - img2) ** 2)
    if mse.item() == 0:
        return float('inf')
    psnr = 20 * torch.log10(torch.tensor(1.0)) - 10 * torch.log10(mse)
    return psnr.item()


def compute_ssim(img1: Tensor, img2: Tensor) -> float:
    """
    Compute SSIM between two single images of shape [1, 1, H, W] or [1, H, W].

    Returns:
        float: SSIM score
    """
    img1_np = img1.squeeze().detach().cpu().numpy()
    img2_np = img2.squeeze().detach().cpu().numpy()
    return ssim(img1_np, img2_np, data_range=1.0)


def compute_ssim_batch(batch1: Tensor, batch2: Tensor) -> float:
    """
    Compute mean SSIM over a batch of grayscale image pairs.

    Args:
        batch1, batch2: shape (N, 1, H, W), values in [0, 1]

    Returns:
        float: average SSIM across batch
    """
    batch1_np = batch1.squeeze(1).detach().cpu().numpy()
    batch2_np = batch2.squeeze(1).detach().cpu().numpy()

    return np.mean([
        ssim(batch1_np[i], batch2_np[i], data_range=1.0)
        for i in range(batch1_np.shape[0])
    ])

#========== Plots ==========
def side_by_side_plot(
    y: Tensor,
    clean_img: Tensor,
    results: Dict[str, Tensor],
    metrics: Dict[str, Tuple[float, float]],
    save_path: str = None
) -> None:
    """
    Display denoising results side by side with PSNR and SSIM annotations.

    Args:
        y (Tensor): Noisy image, shape [1, 1, H, W]
        clean_img (Tensor): Ground truth clean image, shape [1, 1, H, W]
        results (Dict[str, Tensor]): Denoised results by method name.
        metrics (Dict[str, Tuple[float, float]]): PSNR and SSIM values keyed by method name.
        save_path (str, optional): If set, saves the figure to this path.
    """
    titles = ["Noisy Input", "Clean Image"] + list(results.keys())
    images = [y, clean_img] + [results[k] for k in results.keys()]
    total = len(images)

    cols = 4
    rows = math.ceil(total / cols)
    plt.figure(figsize=(4 * cols, 4 * rows))

    for i, (img, title) in enumerate(zip(images, titles)):
        plt.subplot(rows, cols, i + 1)
        img_np = img.squeeze().detach().cpu().numpy()
        plt.imshow(img_np, cmap='gray', vmin=0, vmax=1)
        plt.axis('off')

        if title in metrics:
            psnr, ssim = metrics[title]
            plt.title(f"{title}\nPSNR: {psnr:.2f}, SSIM: {ssim:.3f}")
        else:
            plt.title(title)

    plt.tight_layout()
    if save_path:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        plt.savefig(save_path, dpi=300)
        print(f"📷 Saved side-by-side comparison to: {save_path}")

    plt.show()
#==========================================================================================

# Models

In [5]:
#========== Tikhonov Model ==========
import torch
import torch.nn as nn

class LinearTikhonovDenoiser(nn.Module):
    """
    Args:
        beta (float): Regularization parameter controlling the trade-off
                      between fidelity to the input and regularization.
                      Higher beta values apply stronger denoising.
    """

    def __init__(self, beta: float):
        super().__init__()
        self.scale = 1 / (1 + beta)  # Compute the denoising scale

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass of the denoiser.

        Args:
            x (torch.Tensor): The noisy input tensor.

        Returns:
            torch.Tensor: The denoised output tensor obtained by scaling.
        """
        return self.scale * x  # Apply linear denoising


#========== TV Model ==========
class TVDenoiser(nn.Module):
    def __init__(self, weight: float = 0.1, n_iter: int = 5):
        """
        Args:
            weight (float): Regularization strength (higher → more smoothing)
            n_iter (int): Number of Chambolle iterations
        """
        super().__init__()
        self.weight = weight
        self.n_iter = n_iter

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Args:
            x (torch.Tensor): Image tensor of shape [1, 1, H, W] in [0, 1]

        Returns:
            torch.Tensor: Denoised image of same shape
        """
        x_np = x.squeeze().cpu().numpy()  # shape: H x W
        x_denoised = denoise_tv_chambolle(x_np, weight=self.weight, max_num_iter=self.n_iter)
        return torch.tensor(x_denoised, dtype=x.dtype, device=x.device).unsqueeze(0).unsqueeze(0)

#========== Unet Model ==========
class UNet(nn.Module):
    """
    A simplified 3-level U-Net architecture for grayscale image denoising.

    Architecture:
    - 3 encoding levels with max pooling
    - Bottleneck with 2 conv layers
    - 3 decoding levels with transposed convs and skip connections
    - Final 1x1 convolution to map to single-channel output

    Input:
        Tensor of shape (B, 1, H, W)

    Output:
        Tensor of shape (B, 1, H, W) (same spatial size)
    """
    def __init__(self) -> None:
        super(UNet, self).__init__()

        def conv_block(in_channels: int, out_channels: int) -> nn.Sequential:
            """
            Returns a block with two 3x3 convolutions + ReLU.
            """
            return nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
                nn.ReLU(inplace=True),
                nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
                nn.ReLU(inplace=True),
            )

        # Encoder blocks
        self.enc1 = conv_block(1, 64)
        self.enc2 = conv_block(64, 128)
        self.enc3 = conv_block(128, 256)

        self.pool = nn.MaxPool2d(kernel_size=2)

        # Bottleneck
        self.bottleneck = conv_block(256, 512)

        # Decoder blocks
        self.upconv3 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.dec3 = conv_block(512, 256)

        self.upconv2 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.dec2 = conv_block(256, 128)

        self.upconv1 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.dec1 = conv_block(128, 64)

        # Output layer
        self.final = nn.Conv2d(64, 1, kernel_size=1)

    def forward(self, x: Tensor) -> Tensor:
        """
        Forward pass through the U-Net.

        Args:
            x (Tensor): Input image tensor of shape (B, 1, H, W)

        Returns:
            Tensor: Output tensor of same shape (B, 1, H, W)
        """
        # Encoder
        enc1 = self.enc1(x)
        enc2 = self.enc2(self.pool(enc1))
        enc3 = self.enc3(self.pool(enc2))

        # Bottleneck
        bottleneck = self.bottleneck(self.pool(enc3))

        # Decoder with skip connections
        dec3 = self.upconv3(bottleneck)
        dec3 = self.dec3(torch.cat((dec3, enc3), dim=1))

        dec2 = self.upconv2(dec3)
        dec2 = self.dec2(torch.cat((dec2, enc2), dim=1))

        dec1 = self.upconv1(dec2)
        dec1 = self.dec1(torch.cat((dec1, enc1), dim=1))

        return self.final(dec1)

#========== ViT Model ==========
import torch
import torch.nn as nn

class ViTDenoiser(nn.Module):
    """
    A simple Vision Transformer-based denoiser for grayscale images.

    This model combines a convolutional encoder, a transformer-based
    feature processor, and a convolutional decoder to denoise input images.

    Architecture:
        - Encoder: 2 convolutional layers with ReLU activations.
        - Transformer: 2-layer Transformer Encoder for non-local feature learning.
        - Decoder: 2 convolutional layers to reconstruct the denoised image.

    Notes:
        - Input is assumed to be a grayscale image with shape [B, 1, H, W].
        - Output is a denoised image with the same shape as input.

    """

    def __init__(self):
        super().__init__()

        # Encoder: Converts 1-channel image to 64 feature maps
        self.encoder = nn.Sequential(
            nn.Conv2d(1, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
        )

        # Transformer Encoder: Processes feature sequences
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=64, nhead=8),
            num_layers=2
        )

        # Decoder: Converts 64 feature maps back to 1-channel image
        self.decoder = nn.Sequential(
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 1, kernel_size=3, padding=1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Forward pass of the denoiser.

        Args:
            x (torch.Tensor): Input noisy image tensor of shape [B, 1, H, W].

        Returns:
            torch.Tensor: Denoised output image tensor of shape [B, 1, H, W].
        """
        # Encode the image into deep feature maps
        x = self.encoder(x)  # Shape: [B, 64, H, W]

        # Flatten spatial dimensions and permute for transformer: [H*W, B, C]
        b, c, h, w = x.shape
        x = x.flatten(2).permute(2, 0, 1)

        # Apply transformer encoder
        x = self.transformer(x)

        # Restore original shape: [B, 64, H, W]
        x = x.permute(1, 2, 0).view(b, c, h, w)

        # Decode the feature maps into the denoised image
        x = self.decoder(x)

        return x

#==============================================================================================


# Mathmatical Models


In [6]:
#========== ADMM ==========
def fft_conv2d(x, kernel_fft):
    return torch.real(torch.fft.ifft2(torch.fft.fft2(x) * kernel_fft))

def admm_reconstruct(
    y: torch.Tensor,
    denoiser: torch.nn.Module,
    kernel: torch.Tensor,
    lambda_: float = 0.05,
    rho: float = 0.1,
    max_iter: int = 50,
    device: torch.device = torch.device("cpu"),
    verbose: bool = False
) -> torch.Tensor:
    B, C, H, W = y.shape
    y = y.to(device)
    denoiser = denoiser.to(device).eval()
    kernel = kernel.to(device)

    # === Prepare FFT of PSF ===
    psf = F.pad(kernel, [0, W - kernel.shape[-1], 0, H - kernel.shape[-2]])
    H_fft = torch.fft.fft2(psf)
    H_conj_fft = torch.conj(H_fft)
    Ht_y_fft = H_conj_fft * torch.fft.fft2(y)

    # === Init Variables ===
    x = y.clone()
    v = x.clone()
    u = torch.zeros_like(x)

    for i in range(max_iter):
        # === x-update in Fourier domain ===
        rhs_fft = Ht_y_fft + rho * torch.fft.fft2(v - u)
        denom = H_conj_fft * H_fft + rho
        x = torch.real(torch.fft.ifft2(rhs_fft / denom)).clamp(0, 1)

        # === v-update: denoising step ===
        with torch.no_grad():
            v = denoiser((x + u).clamp(0, 1))

        # === u-update ===
        u = u + x - v

        if verbose and (i % 10 == 0 or i == max_iter - 1):
            err = torch.norm(x - v).item()
            print(f"[ADMM Iter {i+1}/{max_iter}] ‖x−v‖={err:.4f}")

    return x.clamp(0, 1)

#========== RED =========
def apply_blur(x: torch.Tensor, kernel: torch.Tensor) -> torch.Tensor:
    """Applies blur operator H to input x."""
    return F.conv2d(x, kernel, padding="same")


def apply_blur_T(x: torch.Tensor, kernel: torch.Tensor) -> torch.Tensor:
    """Applies transpose of blur operator Hᵀ to input x."""
    return F.conv2d(x, torch.flip(kernel, dims=[2, 3]), padding="same")


def red_sd(
    y: torch.Tensor,
    denoiser: torch.nn.Module,
    kernel: torch.Tensor,
    lambda_: float = 0.05,
    alpha: float = 0.1,
    max_iter: int = 30,
    verbose: bool = True,
    device: str = "cuda" if torch.cuda.is_available() else "cpu",
    x_clean: torch.Tensor = None,  # ✅ NEW: clean reference image
) -> torch.Tensor:
    """
    Perform Regularization by Denoising (RED) image restoration.

    Args:
        y (torch.Tensor): Noisy observed image (shape: [1, 1, H, W]).
        denoiser (torch.nn.Module): Pretrained denoising model (e.g., UNet).
        kernel (torch.Tensor): Convolution kernel for forward model (H).
        lambda_ (float): Regularization strength.
        alpha (float): Step size for gradient update.
        max_iter (int): Number of RED iterations.
        verbose (bool): Whether to print PSNR/SSIM during optimization.
        device (str): 'cuda' or 'cpu'.
        x_clean (torch.Tensor, optional): Clean ground truth image to compare against.

    Returns:
        torch.Tensor: Restored image after RED iterations.
    """
    x = y.clone().detach().to(device)
    y = y.to(device)
    kernel = kernel.to(device)
    denoiser = denoiser.to(device)
    denoiser.eval()

    if x_clean is not None:
        x_clean = x_clean.to(device)

    for i in range(max_iter):
        Hx = apply_blur(x, kernel)
        grad_data = apply_blur_T(Hx - y, kernel)

        with torch.no_grad():
            denoised = denoiser(x)

        grad_prior = x - denoised
        grad = grad_data + lambda_ * grad_prior
        x = x - alpha * grad

        if verbose and (i % 5 == 0 or i == max_iter - 1):
            target = x_clean if x_clean is not None else y
            psnr = compute_psnr(x, target)
            ssim = compute_ssim_batch(x.detach(), target.detach())
            label = "clean" if x_clean is not None else "noisy"
            print(f"[{i+1}/{max_iter}] PSNR ({label}): {psnr:.2f}, SSIM: {ssim:.4f}")

    return x

#========== SMD ==========
def smd_denoise(model, noisy_img, eta, noise_variance, n_steps=1, device="cpu"):
    """
    Perform Score-Matching Denoising (SMD) using a pretrained denoiser model.

    Args:
        model: PyTorch denoiser (e.g., UNet)
        noisy_img (torch.Tensor): shape (1, 1, H, W), normalized to [0,1]
        eta (float): step size
        noise_variance (float): variance of Gaussian noise used in training
        n_steps (int): number of SMD update iterations
        device (str): device for computation

    Returns:
        torch.Tensor: denoised image (same shape as input)
    """
    x = noisy_img.clone().to(device)
    model.eval()

    with torch.no_grad():
        for _ in range(n_steps):
            denoised = model(x)
            score = (denoised - x) / noise_variance
            x = x + eta * score
            x = torch.clamp(x, 0.0, 1.0)

    return x
import torch

def smd_denoise(model, noisy_img, eta, noise_variance, n_steps=1, device="cpu"):
    """
    Perform Score-Matching Denoising (SMD) using a pretrained denoiser model.

    Args:
        model: PyTorch denoiser (e.g., UNet)
        noisy_img (torch.Tensor): shape (1, 1, H, W), normalized to [0,1]
        eta (float): step size
        noise_variance (float): variance of Gaussian noise used in training
        n_steps (int): number of SMD update iterations
        device (str): device for computation

    Returns:
        torch.Tensor: denoised image (same shape as input)
    """
    x = noisy_img.clone().to(device)
    model.eval()

    with torch.no_grad():
        for _ in range(n_steps):
            denoised = model(x)
            score = (denoised - x) / noise_variance
            x = x + eta * score
            x = torch.clamp(x, 0.0, 1.0)

    return x

#============================================================================================

# Training Functions

In [ ]:
def train_validate_test(model: nn.Module,
                        train_loader: DataLoader,
                        val_loader: DataLoader,
                        test_loader: DataLoader,
                        model_name: str,
                        epochs: int = 10,
                        batch_size: int = 8,
                        verbose: bool = False) -> None:
    """
    Trains a denoising model (e.g., UNet) using provided training, validation, and test data loaders.

    Args:
        model (Module): The PyTorch model to train and evaluate.
        train_loader (DataLoader): DataLoader for the training dataset.
        val_loader (DataLoader): DataLoader for the validation dataset.
        test_loader (DataLoader): DataLoader for the test dataset.

    Returns:
        NoReturn: This function does not return anything. It saves training logs, model weights,
        evaluation metrics, and sample output images to the 'output/<model_name>' directory.
    """
    OUTPUT_DIR = Path(f"output/{model_name.lower()}")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    checkpoint_path = OUTPUT_DIR / f"best_{model_name}.pth"
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if verbose:
        print(f"Using device: {device}")
    model = model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.MSELoss()

    best_val_loss = float('inf')
    train_losses, val_losses = [], []
    train_psnrs, val_psnrs = [], []
    train_ssims, val_ssims = [], []

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        train_psnr = 0.0
        train_ssim = 0.0
        loop = tqdm(train_loader, desc=f"Epoch {epoch+1} [Train]", leave=False)
        for noisy, clean in loop:
            noisy, clean = noisy.to(device), clean.to(device)
            output = model(noisy)
            loss = loss_fn(output, clean)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            train_loss += loss.item()
            train_psnr += compute_psnr(output, clean)
            train_ssim += compute_ssim_batch(output.detach(), clean.detach())
            loop.set_postfix(loss=loss.item())

        avg_train_loss = train_loss / len(train_loader)
        avg_train_psnr = train_psnr / len(train_loader)
        avg_train_ssim = train_ssim / len(train_loader)
        train_losses.append(avg_train_loss)
        train_psnrs.append(avg_train_psnr)
        train_ssims.append(avg_train_ssim)

        model.eval()
        val_loss = 0.0
        val_psnr = 0.0
        val_ssim = 0.0
        with torch.no_grad():
            for noisy, clean in tqdm(val_loader, desc=f"Epoch {epoch+1} [Val]", leave=False):
                noisy, clean = noisy.to(device), clean.to(device)
                output = model(noisy)
                val_loss += loss_fn(output, clean).item()
                val_psnr += compute_psnr(output, clean)
                val_ssim += compute_ssim_batch(output.detach(), clean.detach())

        avg_val_loss = val_loss / len(val_loader)
        avg_val_psnr = val_psnr / len(val_loader)
        avg_val_ssim = val_ssim / len(val_loader)
        val_losses.append(avg_val_loss)
        val_psnrs.append(avg_val_psnr)
        val_ssims.append(avg_val_ssim)

        if avg_val_loss < best_val_loss:
            best_val_loss = avg_val_loss
            torch.save(model.state_dict(), checkpoint_path)
            if verbose:
                print(f"✅ Saved better model at epoch {epoch+1}")

        print(f"Epoch {epoch+1}: Train Loss = {avg_train_loss:.6f}, PSNR = {avg_train_psnr:.2f}, SSIM = {avg_train_ssim:.4f} | Val Loss = {avg_val_loss:.6f}, PSNR = {avg_val_psnr:.2f}, SSIM = {avg_val_ssim:.4f}")

    # Plot loss curves
    plt.figure()
    plt.plot(train_losses, label="Train Loss")
    plt.plot(val_losses, label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training vs Validation Loss")
    plt.legend()
    plt.savefig(OUTPUT_DIR / "loss_plot.png")

    # Plot PSNR curves
    plt.figure()
    plt.plot(train_psnrs, label="Train PSNR")
    plt.plot(val_psnrs, label="Val PSNR")
    plt.xlabel("Epoch")
    plt.ylabel("PSNR (dB)")
    plt.title("Training vs Validation PSNR")
    plt.legend()
    plt.savefig(OUTPUT_DIR / "psnr_plot.png")

    # Plot SSIM curves
    plt.figure()
    plt.plot(train_ssims, label="Train SSIM")
    plt.plot(val_ssims, label="Val SSIM")
    plt.xlabel("Epoch")
    plt.ylabel("SSIM")
    plt.title("Training vs Validation SSIM")
    plt.legend()
    plt.savefig(OUTPUT_DIR / "ssim_plot.png")

    # Test Phase
    model.eval()
    test_loss = 0.0
    test_psnr = 0.0
    test_ssim = 0.0
    with torch.no_grad():
        for noisy, clean in tqdm(test_loader, desc="Testing", leave=False):
            noisy, clean = noisy.to(device), clean.to(device)
            output = model(noisy)
            test_loss += loss_fn(output, clean).item()
            test_psnr += compute_psnr(output, clean)
            test_ssim += compute_ssim_batch(output.detach(), clean.detach())

    print(f"Test Loss: {test_loss / len(test_loader):.6f}, PSNR: {test_psnr / len(test_loader):.2f} dB, SSIM: {test_ssim / len(test_loader):.4f}")

    # Save example output triplet (noisy, output, clean) with labels
    noisy, clean = next(iter(test_loader))
    model.eval()
    with torch.no_grad():
        output = model(noisy.to(device)).cpu()

    fig, axes = plt.subplots(3, batch_size, figsize=(batch_size * 2, 6))
    row_titles = ["Noisy", "Denoised", "Clean"]
    for row, title in enumerate(row_titles):
        for col in range(batch_size):
            axes[row, col].imshow([
                noisy, output, clean
            ][row][col][0], cmap='gray')
            axes[row, col].axis('off')
            if col == 0:
                axes[row, col].set_ylabel(title, fontsize=12)

    fig.suptitle("First row: Noisy | Second row: Denoised | Third row: Clean", fontsize=16)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "sample_results.png")
    plt.close()

#============================================================================================

# Inferences

In [11]:
def Red_Model_Based(models: Dict[str, nn.Module],
                    train_loader: DataLoader,
                    val_loader: DataLoader,
                    test_loader: DataLoader,
                    device: str = "cpu",
                    verbose: bool = False,
                    n_images: int = 6):

    psf_np = np.load("Data/psf.npy")
    psf_np /= psf_np.sum()
    psf_tensor = torch.tensor(psf_np, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)

    results = {}
    metrics = {}

    for idx, (noisy_img, clean_img) in enumerate(test_loader):
        if idx >= n_images:
            break
        noisy_img = noisy_img.to(device)
        clean_img = clean_img.to(device)

        with torch.no_grad():
            for model_name, model in models.items():
                output = model(noisy_img)
                psnr = compute_psnr(output, clean_img)
                ssim = compute_ssim(output, clean_img)
                key = f"[{idx}] Direct-{model_name}"
                results[key] = output
                metrics[key] = (psnr, ssim)
                print(f"🧠 {key} — PSNR: {psnr:.2f}, SSIM: {ssim:.4f}")

        denoisers = {
            **{f"RED-{name}": model for name, model in models.items()},
            "RED-TV": TVDenoiser(weight=0.1, n_iter=5),
            "RED-Tikhonov": LinearTikhonovDenoiser(beta=0.05)
        }

        print(f"\n🔬 Running RED on image {idx+1}...")
        for name, denoiser in denoisers.items():
            print(f"Running: {name}...")
            is_learned_model = name.startswith("RED-") and name.split("RED-")[1] in models
            x_restored = red_sd(
                y=noisy_img,
                denoiser=denoiser,
                kernel=psf_tensor,
                lambda_=0.05,
                alpha=0.1,
                max_iter=50,
                x_clean=clean_img,
                verbose=verbose,
                device=device
            )
            key = f"[{idx}] {name}"
            results[key] = x_restored
            psnr = compute_psnr(x_restored, clean_img)
            ssim = compute_ssim(x_restored, clean_img)
            metrics[key] = (psnr, ssim)
            print(f"✅ {key} — PSNR: {psnr:.2f}, SSIM: {ssim:.4f}")

    side_by_side_plot(noisy_img, clean_img, results, metrics, save_path="output/red_results/red_comparison_all.png")

    with open("output/red_results/red_metrics_all.json", "w") as f:
        json.dump({k: [float(v[0]), float(v[1])] for k, v in metrics.items()}, f, indent=2)


def ADMM_Model_Based(models: Dict[str, nn.Module],
                     train_loader: DataLoader,
                     val_loader: DataLoader,
                     test_loader: DataLoader,
                     device: str = "cpu",
                     verbose: bool = False,
                     n_images: int = 6):

    psf_np = np.load("Data/psf.npy")
    psf_np /= psf_np.sum()
    psf_tensor = torch.tensor(psf_np, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)

    denoisers = {
        **{f"ADMM-{name}": model for name, model in models.items()},
        "ADMM-TV": TVDenoiser(weight=0.1, n_iter=5),
        "ADMM-Tikhonov": LinearTikhonovDenoiser(beta=0.05)
    }


    for idx, (noisy_img, clean_img) in enumerate(test_loader):
        if idx >= n_images:
            break
        noisy_img = noisy_img.to(device)
        clean_img = clean_img.to(device)

        results = {}
        metrics = {}

        print(f"\n🧠 Running ADMM on image {idx+1}...")
        for name, denoiser in denoisers.items():
            x_restored = admm_reconstruct(
                y=noisy_img,
                denoiser=denoiser,
                kernel=psf_tensor,
                lambda_=0.001,
                rho=0.1,
                max_iter=50,
                verbose=verbose,
                device=device
            )
            key = f"[{idx}] {name}"
            results[key] = x_restored
            psnr = compute_psnr(x_restored, clean_img)
            ssim = compute_ssim(x_restored, clean_img)
            metrics[key] = (psnr, ssim)
            print(f"✅ {key} — PSNR: {psnr:.2f}, SSIM: {ssim:.4f}")

    Path("output/admm_results").mkdir(parents=True, exist_ok=True)
    side_by_side_plot(noisy_img, clean_img, results, metrics, save_path="output/admm_results/admm_comparison_all.png")
    with open("output/admm_results/admm_metrics_all.json", "w") as f:
        json.dump({k: [float(v[0]), float(v[1])] for k, v in metrics.items()}, f, indent=2)


def SMD_Model_Based(models: Dict[str, nn.Module],
                    train_loader: DataLoader,
                    val_loader: DataLoader,
                    test_loader: DataLoader,
                    noise_var: float,
                    noise_std: float = 0.05,
                    smd_eta: float = 0.0003,
                    smd_steps: int = 10,
                    device: str = "cpu",
                    verbose: bool = False,
                    n_images: int = 6):

    os.makedirs(f"output/smd_results", exist_ok=True)

    psf_np = np.load("Data/psf.npy")
    psf_tensor = torch.tensor(psf_np, dtype=torch.float32).unsqueeze(0).unsqueeze(0).to(device)

    for idx, (noisy_img, clean_img) in enumerate(test_loader):
        if idx >= n_images:
            break
        noisy_img = noisy_img.to(device)
        clean_img = clean_img.to(device)

        results = {}
        metrics = {}

        print(f"\n🧪 Running SMD on image {idx+1}...")
        for name, model in models.items():
            model.eval()
            smd_restored = smd_denoise(
                model=model.to(device),
                noisy_img=noisy_img,
                eta=smd_eta,
                noise_variance=noise_var,
                n_steps=smd_steps,
                device=device
            )

            key = f"[{idx}] SMD-{name}"
            results[key] = smd_restored
            psnr = compute_psnr(smd_restored, clean_img)
            ssim = compute_ssim(smd_restored, clean_img)
            metrics[key] = (psnr, ssim)

            if verbose:
                print(f"✅ {key} — PSNR: {psnr:.2f}, SSIM: {ssim:.4f}")

    side_by_side_plot(noisy_img, clean_img, results, metrics, save_path="output/smd_results/smd_comparison_all.png")
    with open("output/smd_results/smd_metrics.json", "w") as f:
        json.dump({k: [float(v[0]), float(v[1])] for k, v in metrics.items()}, f, indent=2)


In [ ]:
#======================================= Main ===========================================
# === CONFIGURATION ===
VERBOSE = True
DO_TRAIN = False
DOWNLOAD_AND_PROCESS_SDSS = False # change to True if no data downloaded
BATCH_SIZE = 8
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NOISE_STD = 0.05
NOISE_VARIANCE = NOISE_STD ** 2
SMD_ETA = 0.0003
SMD_STEPS = 10
VIT_PATH = Path(f"output/vit/best_vit.pth")
UNET_PATH = Path(f"output/unet/best_unet.pth")

if DOWNLOAD_AND_PROCESS_SDSS:
    download_and_process_sdss()

# === Load Dataset ===
dataset = AstroDenoisingDataset("Data/clean", "Data/noisy", transform=transforms.ToTensor())
train_size = int(0.8 * len(dataset))
val_size = int(0.1 * len(dataset))
test_size = len(dataset) - train_size - val_size
train_dataset, val_dataset, test_dataset = random_split(dataset, [train_size, val_size, test_size])

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

# Define models
unet_model = UNet().to(DEVICE)
vit_model = ViTDenoiser().to(DEVICE)

models = {"vit": vit_model,
          "unet": unet_model
          }

# Training
if DO_TRAIN:
    train_validate_test(vit_model, train_loader, val_loader, test_loader, "ViT",
                        epochs=10, batch_size=BATCH_SIZE, verbose=VERBOSE)
    train_validate_test(unet_model, train_loader, val_loader, test_loader, "unet",
                        epochs=8, batch_size=BATCH_SIZE, verbose=VERBOSE)

if not (VIT_PATH.exists() or UNET_PATH.exists()):
    raise FileNotFoundError(f"No model checkpoint found for ViT or unet")

unet_model.load_state_dict(torch.load(UNET_PATH, map_location=DEVICE))
vit_model.load_state_dict(torch.load(VIT_PATH, map_location=DEVICE))

if VERBOSE:
    print(f"✅ Loaded pre-trained models")

Red_Model_Based(models, train_loader, val_loader, test_loader, DEVICE, VERBOSE)
ADMM_Model_Based(models, train_loader, val_loader, test_loader, DEVICE, VERBOSE)
SMD_Model_Based(models, train_loader, val_loader, test_loader, NOISE_VARIANCE, NOISE_STD, SMD_ETA,SMD_STEPS, DEVICE, VERBOSE)


✅ Loaded pre-trained models
🧠 [0] Direct-vit — PSNR: 25.29, SSIM: 0.2471
🧠 [0] Direct-unet — PSNR: 39.39, SSIM: 0.8873

🔬 Running RED on image 1...
Running: RED-vit...
[1/50] PSNR (clean): 28.44, SSIM: 0.3741
[6/50] PSNR (clean): 28.50, SSIM: 0.3765
[11/50] PSNR (clean): 28.56, SSIM: 0.3801
[16/50] PSNR (clean): 28.63, SSIM: 0.3842
[21/50] PSNR (clean): 28.69, SSIM: 0.3886
[26/50] PSNR (clean): 28.75, SSIM: 0.3930
[31/50] PSNR (clean): 28.81, SSIM: 0.3972
[36/50] PSNR (clean): 28.87, SSIM: 0.4013
[41/50] PSNR (clean): 28.92, SSIM: 0.4052
[46/50] PSNR (clean): 28.97, SSIM: 0.4089
[50/50] PSNR (clean): 29.00, SSIM: 0.4117
✅ [0] RED-vit — PSNR: 29.00, SSIM: 0.4117
Running: RED-unet...
[1/50] PSNR (clean): 28.46, SSIM: 0.3759
[6/50] PSNR (clean): 28.61, SSIM: 0.3853
[11/50] PSNR (clean): 28.74, SSIM: 0.3935
[16/50] PSNR (clean): 28.86, SSIM: 0.4010
[21/50] PSNR (clean): 28.97, SSIM: 0.4079
[26/50] PSNR (clean): 29.08, SSIM: 0.4143
[31/50] PSNR (clean): 29.18, SSIM: 0.4205
[36/50] PSNR (cle